In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import os
from dotenv import load_dotenv
from datetime import datetime

print("All libraries loaded!")

In [ ]:
np.random.seed(42)

# Generate 30 employees
n_employees = 30
n_days = 30

employees = []
for i in range(1, n_employees + 1):
    employee_id = f"EMP{i:03d}"
    
    # Randomly assign attendance patterns
    pattern = np.random.choice(['good', 'average', 'poor'], p=[0.5, 0.3, 0.2])
    
    if pattern == 'good':
        attendance_rate = np.random.uniform(0.90, 1.0)
        late_rate = np.random.uniform(0.0, 0.1)
        productivity = np.random.uniform(75, 100)
    elif pattern == 'average':
        attendance_rate = np.random.uniform(0.75, 0.90)
        late_rate = np.random.uniform(0.1, 0.2)
        productivity = np.random.uniform(55, 75)
    else:
        attendance_rate = np.random.uniform(0.50, 0.75)
        late_rate = np.random.uniform(0.2, 0.4)
        productivity = np.random.uniform(30, 55)
    
    days_present = int(n_days * attendance_rate)
    days_absent = n_days - days_present
    days_late = int(days_present * late_rate)
    
    employees.append({
        'Employee_ID': employee_id,
        'Days_Present': days_present,
        'Days_Absent': days_absent,
        'Days_Late': days_late,
        'Productivity_Score': round(productivity, 1),
        'Attendance_Rate': round(attendance_rate * 100, 1),
        'Pattern': pattern
    })

df = pd.DataFrame(employees)
print(df.head(10))
print(f"\nTotal employees: {len(df)}")

In [ ]:
# Define thresholds
ATTENDANCE_THRESHOLD = 80  # Below 80% is flagged
LATE_THRESHOLD = 5         # More than 5 late days is flagged
PRODUCTIVITY_THRESHOLD = 60 # Below 60 productivity is flagged

# Flag employees with issues
df['Attendance_Flag'] = df['Attendance_Rate'] < ATTENDANCE_THRESHOLD
df['Late_Flag'] = df['Days_Late'] > LATE_THRESHOLD
df['Productivity_Flag'] = df['Productivity_Score'] < PRODUCTIVITY_THRESHOLD

# Overall risk flag
df['At_Risk'] = (df['Attendance_Flag'] | df['Late_Flag'] | df['Productivity_Flag'])

# Risk level
def get_risk_level(row):
    flags = sum([row['Attendance_Flag'], row['Late_Flag'], row['Productivity_Flag']])
    if flags == 0:
        return 'Low'
    elif flags == 1:
        return 'Medium'
    else:
        return 'High'

df['Risk_Level'] = df.apply(get_risk_level, axis=1)

print(df[['Employee_ID', 'Attendance_Rate', 'Days_Late', 'Productivity_Score', 'Risk_Level']].head(10))
print(f"\nAt Risk Employees: {df['At_Risk'].sum()}")
print(f"High Risk: {len(df[df['Risk_Level'] == 'High'])}")
print(f"Medium Risk: {len(df[df['Risk_Level'] == 'Medium'])}")
print(f"Low Risk: {len(df[df['Risk_Level'] == 'Low'])}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Chart 1 - Attendance Rate Distribution
axes[0, 0].hist(df['Attendance_Rate'], bins=10, color='steelblue', edgecolor='white')
axes[0, 0].axvline(x=ATTENDANCE_THRESHOLD, color='red', linestyle='--', label='Threshold (80%)')
axes[0, 0].set_title('Attendance Rate Distribution')
axes[0, 0].set_xlabel('Attendance Rate (%)')
axes[0, 0].set_ylabel('Number of Employees')
axes[0, 0].legend()

# Chart 2 - Productivity Score Distribution
axes[0, 1].hist(df['Productivity_Score'], bins=10, color='steelblue', edgecolor='white')
axes[0, 1].axvline(x=PRODUCTIVITY_THRESHOLD, color='red', linestyle='--', label='Threshold (60)')
axes[0, 1].set_title('Productivity Score Distribution')
axes[0, 1].set_xlabel('Productivity Score')
axes[0, 1].set_ylabel('Number of Employees')
axes[0, 1].legend()

# Chart 3 - Risk Level Count
risk_counts = df['Risk_Level'].value_counts()
colors = {'Low': 'green', 'Medium': 'orange', 'High': 'red'}
axes[1, 0].bar(risk_counts.index, risk_counts.values, 
               color=[colors[r] for r in risk_counts.index])
axes[1, 0].set_title('Employee Risk Level Distribution')
axes[1, 0].set_xlabel('Risk Level')
axes[1, 0].set_ylabel('Number of Employees')

# Chart 4 - Attendance vs Productivity Scatter
colors_scatter = {'Low': 'green', 'Medium': 'orange', 'High': 'red'}
for risk in ['Low', 'Medium', 'High']:
    subset = df[df['Risk_Level'] == risk]
    axes[1, 1].scatter(subset['Attendance_Rate'], subset['Productivity_Score'],
                       label=risk, color=colors_scatter[risk], s=100)
axes[1, 1].axvline(x=ATTENDANCE_THRESHOLD, color='red', linestyle='--', alpha=0.5)
axes[1, 1].axhline(y=PRODUCTIVITY_THRESHOLD, color='red', linestyle='--', alpha=0.5)
axes[1, 1].set_title('Attendance vs Productivity')
axes[1, 1].set_xlabel('Attendance Rate (%)')
axes[1, 1].set_ylabel('Productivity Score')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Train ML model to predict risk
X = df[['Days_Present', 'Days_Absent', 'Days_Late', 'Productivity_Score', 'Attendance_Rate']]
y = df['Risk_Level']

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Train Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Accuracy
accuracy = model.score(X_test, y_test)
print(f"Model Accuracy: {round(accuracy * 100, 1)}%")

# Predict on full dataset
df['Predicted_Risk'] = le.inverse_transform(model.predict(X))
print("\nPredictions vs Actual:")
print(df[['Employee_ID', 'Risk_Level', 'Predicted_Risk']].head(10))

In [ ]:
def send_manager_report(df):
    
    load_dotenv(r'C:\Users\Ayaan\deadline-reminder\.env')
    
    sender_email = os.getenv('EMAIL')
    receiver_email = os.getenv('EMAIL')
    app_password = os.getenv('EMAIL_PASSWORD')
    
    high_risk = df[df['Risk_Level'] == 'High']
    medium_risk = df[df['Risk_Level'] == 'Medium']
    low_risk = df[df['Risk_Level'] == 'Low']
    
    subject = f"📊 Weekly Employee Attendance & Productivity Report — {datetime.now().strftime('%B %d, %Y')}"
    
    body = f"""
    Dear Manager,

    Here is your weekly Employee Attendance & Productivity Report.

    📊 SUMMARY:
    • Total Employees: {len(df)}
    • High Risk: {len(high_risk)} employees
    • Medium Risk: {len(medium_risk)} employees  
    • Low Risk: {len(low_risk)} employees

    🚨 HIGH RISK EMPLOYEES (Immediate Attention Required):
    """
    
    for _, row in high_risk.iterrows():
        body += f"""
    • {row['Employee_ID']}
      - Attendance Rate: {row['Attendance_Rate']}%
      - Days Absent: {row['Days_Absent']}
      - Days Late: {row['Days_Late']}
      - Productivity Score: {row['Productivity_Score']}
    """
    
    body += f"""
    ⚠️ MEDIUM RISK EMPLOYEES (Monitor Closely):
    """
    
    for _, row in medium_risk.iterrows():
        body += f"\n    • {row['Employee_ID']} — Attendance: {row['Attendance_Rate']}% | Productivity: {row['Productivity_Score']}"
    
    body += f"""

    ✅ LOW RISK EMPLOYEES: {len(low_risk)} employees performing well.

    This is an automated report from the Employee Tracker System.
    """
    
    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = receiver_email
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))
    
    with smtplib.SMTP('smtp.gmail.com', 587) as server:
        server.ehlo()
        server.starttls()
        server.login(sender_email, app_password)
        server.sendmail(sender_email, receiver_email, msg.as_string())
    
    print("✅ Manager report sent successfully!")

print("Email report function ready!")

In [ ]:
send_manager_report(df)

In [ ]:
send_manager_report(df)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(r'C:\Users\Ayaan\deadline-reminder\.env', override=True)

print("EMAIL:", os.getenv('EMAIL'))
print("PASSWORD loaded:", os.getenv('EMAIL_PASSWORD') is not None)

In [ ]:
send_manager_report(df)

In [11]:
import os
print(os.getcwd())

C:\Users\Ayaan\employee-tracker
